# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
!pip install duckdb --quiet

import duckdb
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [13]:
df_schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet') LIMIT 1").df()
for name, dtype in zip(df_schema['column_name'], df_schema['column_type']):
    print(f"{name:<25} {dtype}")

client_hash_id            VARCHAR
content_hash_id           VARCHAR
keyword_hash_id           VARCHAR
url_hash_id               VARCHAR
keyword_char_count        BIGINT
keyword_token_count       BIGINT
url_char_count            BIGINT
content_created_date      DATE
content_updated_date      DATE
content_type              VARCHAR
search_volume             BIGINT
competition               DOUBLE
competition_level         VARCHAR
cpc                       DOUBLE
main_intent               VARCHAR
backlinks                 BIGINT
category_count            BIGINT
keyword_created_date      DATE
provider_used             VARCHAR
model_used                VARCHAR
char_count                BIGINT
word_count                BIGINT
last_optimized_date       DATE
optimization_eligible_date DATE
is_published              BOOLEAN
is_deleted                BOOLEAN


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1: Staleness (days since content_updated_date >= 180, leak-corrected)

Bucket table:
  fresh    n=647,333    avg_clicks=0.121
  stale    n=1,412      avg_clicks=0.019
  unknown  n=2,962,316  avg_clicks=0.251

Verdict: CONFIRMED (direction holds, but coverage is limited)

After removing rows where content_updated_date fell after report_date
(future-dated updates — a real leakage risk caught during this check),
fresh pages still get ~6x more clicks than stale pages, confirming the
expected direction. Important limitation: 82% of rows fall into
"unknown" — dim_content appears to store only each page's current,
latest update date, not a full history, so staleness can't be honestly
determined for most of the panel. This signal is real but only usable
on a minority of rows; the rule should treat "unknown" as its own
category, not silently as "fresh."

Signal 2: CTR vs. position (top10 vs. below10 ranking)

Bucket table:
  top10     n=2,183,484   avg_ctr=0.0039 (0.39%)
  below10   n=1,427,577   avg_ctr=0.0018 (0.18%)

Verdict: CONFIRMED

Top-10 ranked pages get roughly 2.1x the CTR of pages ranked below 10 —
confirms the expected relationship between position and CTR. This gives
a real baseline: a page ranking in the top 10 but showing CTR well below
~0.39% is underperforming for its position, which is exactly the
"CTR-fix" pattern the rule should flag. Unlike staleness, both buckets
here have large, comparable sample sizes (2.18M vs 1.43M), so this
signal is both confirmed and well-supported, not thin.

a page is worth reviewing if its stale and its visible enough to matter and its CTR is underperforming for its ranked position.
stale_low_ctr — stale, and CTR is below the position-appropriate benchmark
stale_visible — stale and getting real impressions, but CTR wasn't checked/didn't apply
not_flagged — didn't meet the staleness + visibility bar

In [14]:
staleness_check = con.sql(f"""
    SELECT
        CASE
            WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) < 0 THEN 'unknown'
            WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) >= 180 THEN 'stale'
            ELSE 'fresh'
        END AS bucket,
        COUNT(*) AS n,
        AVG(f.gsc_clicks) AS avg_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON f.content_hash_id = c.content_hash_id AND f.client_hash_id = c.client_hash_id
    WHERE f.gsc_data_available IS TRUE
    GROUP BY bucket
""")
staleness_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────┬──────────────────────┐
│ bucket  │    n    │      avg_clicks      │
│ varchar │  int64  │        double        │
├─────────┼─────────┼──────────────────────┤
│ fresh   │  647333 │  0.12068595297937847 │
│ stale   │    1412 │ 0.019121813031161474 │
│ unknown │ 2962316 │   0.2510471536459986 │
└─────────┴─────────┴──────────────────────┘

In [15]:
con.sql(f"""
    SELECT MIN(content_updated_date), MAX(content_updated_date), COUNT(*)
    FROM read_parquet('{rel}/dim_content.parquet')
""")

┌───────────────────────────┬───────────────────────────┬──────────────┐
│ min(content_updated_date) │ max(content_updated_date) │ count_star() │
│           date            │           date            │    int64     │
├───────────────────────────┼───────────────────────────┼──────────────┤
│ 2024-10-28                │ 2026-07-06                │       519606 │
└───────────────────────────┴───────────────────────────┴──────────────┘

In [16]:
ctr_position_check = con.sql(f"""
    SELECT
        CASE WHEN gsc_avg_position <= 10 THEN 'top10' ELSE 'below10' END AS bucket,
        COUNT(*) AS n,
        AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END) AS avg_ctr
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY bucket
""")
ctr_position_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────┬──────────────────────┐
│ bucket  │    n    │       avg_ctr        │
│ varchar │  int64  │        double        │
├─────────┼─────────┼──────────────────────┤
│ top10   │ 2183484 │ 0.003899988453596719 │
│ below10 │ 1427577 │ 0.001827718018398195 │
└─────────┴─────────┴──────────────────────┘

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
baseline_df = con.sql(f"""
    SELECT
        f.report_date, f.client_hash_id, f.content_hash_id,
        f.gsc_impressions, f.gsc_clicks, f.gsc_avg_position,
        c.content_updated_date,
        DATE_DIFF('day', c.content_updated_date, f.report_date) AS days_since_update
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON f.content_hash_id = c.content_hash_id AND f.client_hash_id = c.client_hash_id
    WHERE f.gsc_data_available IS TRUE AND f.gsc_impressions > 0
""").df()

baseline_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,content_updated_date,days_since_update
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,2026-05-18,-78
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,2026-05-18,-78
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,2026-07-06,-127
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,2026-05-18,-78
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,2026-05-18,-78


In [18]:


# content_updated_date sometimes falls AFTER report_date -- that would leak
# future info into a point-in-time signal. Treat those as unknown, not stale.
baseline_df.loc[baseline_df['days_since_update'] < 0, 'days_since_update'] = None

baseline_df['days_since_update'].describe()

,days_since_update
count,648745.000000
mean,21.049269
std,17.939225
min,0.000000
25%,11.000000
50%,19.000000
75%,27.000000
max,303.000000


In [19]:
import pandas as pd
import numpy as np

# CTR and expected CTR benchmark (vectorized, no row-by-row loop)
baseline_df['ctr'] = np.where(
    baseline_df['gsc_impressions'] > 0,
    baseline_df['gsc_clicks'] / baseline_df['gsc_impressions'],
    0
)
expected_ctr = np.where(baseline_df['gsc_avg_position'] <= 10, 0.0039, 0.0018)
ctr_underperforming = baseline_df['ctr'] < (expected_ctr * 0.5)

# Staleness bucket
days = baseline_df['days_since_update']
is_stale = days >= 180
is_unknown = days.isna()

visible = baseline_df['gsc_impressions'] >= 10

# Build score, reason_code, action using np.select -- vectorized, one pass
conditions = [
    is_stale & visible & ctr_underperforming,
    is_stale & visible,
    ctr_underperforming & visible,
]
scores = [
    baseline_df['gsc_impressions'],
    baseline_df['gsc_impressions'] * 0.5,
    baseline_df['gsc_impressions'] * 0.3,
]
reason_codes = ['stale_low_ctr', 'stale_visible', 'ctr_underperforming']
actions = ['review_and_refresh', 'review', 'review_ctr']

baseline_df['score'] = np.select(conditions, scores, default=0)
baseline_df['reason_code'] = np.select(conditions, reason_codes, default='not_flagged')
baseline_df['action'] = np.select(conditions, actions, default='no_action')

baseline_df[baseline_df['score'] > 0].sort_values('score', ascending=False).head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,content_updated_date,days_since_update,ctr,score,reason_code,action
1868711,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.083350,2026-06-11,NaN,0.000025,12025.2,ctr_underperforming,review_ctr
42750,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,2.764916,2026-07-04,NaN,0.000051,11700.9,ctr_underperforming,review_ctr
42797,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,2026-07-04,NaN,0.000000,11210.4,ctr_underperforming,review_ctr
1699915,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.181500,2026-06-22,NaN,0.000000,10014.9,ctr_underperforming,review_ctr
1670579,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.132532,2026-06-11,NaN,0.000000,9887.4,ctr_underperforming,review_ctr
1693198,2026-03-29,client_23a62021009f63c4,content_44f34c0a90047651,32756,2,0.142508,2026-06-11,NaN,0.000061,9826.8,ctr_underperforming,review_ctr
1823880,2026-03-31,client_73cda7b4e4f265ea,content_fec55986a1868d62,31472,0,0.083407,2026-06-22,NaN,0.000000,9441.6,ctr_underperforming,review_ctr
3248047,2026-03-25,client_23a62021009f63c4,content_44f34c0a90047651,30964,1,0.117814,2026-06-11,NaN,0.000032,9289.2,ctr_underperforming,review_ctr
3212732,2026-03-24,client_23a62021009f63c4,content_44f34c0a90047651,30791,2,0.088955,2026-06-11,NaN,0.000065,9237.3,ctr_underperforming,review_ctr
1811914,2026-03-26,client_23a62021009f63c4,content_44f34c0a90047651,30573,2,0.238315,2026-06-11,NaN,0.000065,9171.9,ctr_underperforming,review_ctr


In [20]:
import os
os.makedirs('work/outputs', exist_ok=True)
baseline_df.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(baseline_df)} rows to work/outputs/baseline_action_score.csv")

Wrote 3611061 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Rows 1, 5, 6, 8, 9, 10 — content_44f34c0a90047651 (client_23a62021009f63c4),
6 separate March days, 30,573–40,084 impressions each, CTR 0.000000–0.000065.
Action: review_ctr. Why: massively underperforming CTR at huge volume.
What would make it wrong: this isn't normal content underperformance — a
near-zero CTR at 30k+ impressions, repeated day after day for the same page,
looks like a tracking or indexing bug (broken link in the SERP snippet,
noindex tag, or bot-inflated impressions), not a content-quality problem.
Reviewing this as "rewrite the content" would be the wrong action; it needs
a technical check first.

Rows 2, 13 — content_34a70fea29d15f24 (client_62f4a7e64f5e0096),
27,410–39,003 impressions, 0–2 clicks. Action: review_ctr. Why: same
near-zero CTR pattern as above. What would make it wrong: same technical-
issue risk — worth checking if this URL actually resolves correctly before
assuming it's a content problem.

Row 3 — content_945d6ff91386c817, 37,368 impressions, 0 clicks. Action:
review_ctr. Why: zero clicks at high volume. What would make it wrong:
single-day snapshot only (unlike the repeating pattern above) — could be a
one-off dip rather than a persistent issue; worth checking if this page
appears on other days before prioritizing it as high as the repeat offenders.

Rows 4, 7 — content_fec55986a1868d62 (client_73cda7b4e4f265ea), 31,472–33,383
impressions, 0 clicks both days. Action: review_ctr. Why: same zero-CTR
pattern. What would make it wrong: same technical-check concern as the
first group.

Rows 11, 12, 17 — content_9c057b66c30a3abb (client_73cda7b4e4f265ea),
3 separate days, 24,233–28,973 impressions, 0 clicks every time. Action:
review_ctr. Why: consistent zero-click pattern across multiple days, same
client as the fec55986 pages above — worth checking if this whole client's
tracking setup has a shared issue, not just individual pages.

Row 14, 16 — content_0e03de7680314cd5 (client_e547b89c05043229),
24,335–25,582 impressions, 46–49 clicks, CTR ~0.19%. Action: review_ctr.
Why: this one is a genuinely different case — it DOES have real clicks,
just a low CTR relative to the benchmark. What would make it wrong: this
looks like an actual content/snippet quality issue, not a tracking bug —
the most legitimate "review the content" pick in the top 20 so far.

Row 15 — content_1642f339bd6e7c8d, 24,456 impressions, 1 click. Action:
review_ctr. Why: near-zero CTR at real volume. What would make it wrong:
only one click ever recorded is closer to the "possible tracking issue"
group than the genuine-low-CTR group — worth checking impressions source
before trusting this number.

Rows 18, 20 — content_8d7d99f109e19aa2 (client_e547b89c05043229),
21,555–22,321 impressions, 3–4 clicks, CTR ~0.014-0.018%. Action:
review_ctr. Why: low but non-zero CTR, same client as the more legitimate
0e03de case above. What would make it wrong: still quite low click counts
to draw a strong conclusion from — worth a slightly longer time window
before treating this as confirmed.

Row 19 — content_6a9c79f55413b447, 21,998 impressions, 5 clicks, CTR ~0.023%.
Action: review_ctr. Why: low CTR at real volume, real (if small) clicks.
What would make it wrong: same "small click count, need more data" caveat
as above.

In [21]:
top20 = baseline_df[baseline_df['score'] > 0].sort_values('score', ascending=False).head(20)
top20[['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'ctr', 'reason_code', 'action', 'score']]

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,reason_code,action,score
1868711,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.000025,ctr_underperforming,review_ctr,12025.2
42750,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,0.000051,ctr_underperforming,review_ctr,11700.9
42797,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,0.000000,ctr_underperforming,review_ctr,11210.4
1699915,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.000000,ctr_underperforming,review_ctr,10014.9
1670579,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.000000,ctr_underperforming,review_ctr,9887.4
1693198,2026-03-29,client_23a62021009f63c4,content_44f34c0a90047651,32756,2,0.000061,ctr_underperforming,review_ctr,9826.8
1823880,2026-03-31,client_73cda7b4e4f265ea,content_fec55986a1868d62,31472,0,0.000000,ctr_underperforming,review_ctr,9441.6
3248047,2026-03-25,client_23a62021009f63c4,content_44f34c0a90047651,30964,1,0.000032,ctr_underperforming,review_ctr,9289.2
3212732,2026-03-24,client_23a62021009f63c4,content_44f34c0a90047651,30791,2,0.000065,ctr_underperforming,review_ctr,9237.3
1811914,2026-03-26,client_23a62021009f63c4,content_44f34c0a90047651,30573,2,0.000065,ctr_underperforming,review_ctr,9171.9


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:

1. The suspected tracking/indexing issue pages (content_44f34c0a90047651,
content_34a70fea29d15f24, content_fec55986a1868d62, content_9c057b66c30a3abb)
— 6+ of the top 20 rows. Near-zero CTR at 25,000-40,000 impressions, repeated
across multiple March days for the same page. This pattern is too extreme
and too persistent to be normal content underperformance. Treating these as
"needs a content rewrite" would likely be wrong — they need a technical
check first (does the URL resolve, is there a noindex tag, is impression
data possibly inflated by bots) before any content action makes sense.

2. Single-day-only picks with 0 clicks (e.g. content_945d6ff91386c817) —
weaker than the repeating pattern above, since a one-day zero-click dip
could be normal noise rather than a real problem. These shouldn't be
weighted the same as a page that repeats the same failure across a week.

3. The rule never surfaces its own highest-priority reason code
(stale_low_ctr) in the top 20, because genuinely stale content is rare
(n=1,412 rows) relative to the ctr_underperforming pattern. This means the
current score weighting effectively buries staleness entirely under raw
impression volume — a real weakness in the baseline as built, not a data
problem.

Leakage check:

- content_updated_date was found to sometimes fall AFTER report_date, which
would have let future information leak into a point-in-time staleness
signal. Caught during Signal 1, fixed by treating any row where the update
date is after the report date as "unknown" rather than "fresh" or "stale"
(confirmed in baseline_df: these rows now show NaN for days_since_update,
not negative numbers).
- No product/business decision flags (e.g. manual override columns,
"already reviewed" flags) exist in the fact or dim tables used here, so
there's no risk of a product-decision flag leaking into the score.
- No columns from outside the March 2026 window were used in scoring — all
inputs (impressions, clicks, position, updated date) are from the same
report_date row or a fixed reference point (content_updated_date, gated by
the fix above).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.